In [0]:
# ─────────────────────────────────────────────
# Notebook:  03_gold_aggregate
# Purpose:   Daily revenue aggregation by store
# Author:    oakville3456
# Branch:    feature/add-comments
# Updated:   2026-06-04
# ─────────────────────────────────────────────
from pyspark.sql import functions as F

# ── read Silver by TABLE NAME ────────────────────────
silver = spark.read.table("adb_retail_dev.silver.sales")

# ── aggregate ───────────────────────────────────────
gold = (
    silver
    .groupBy("store_id", "order_date")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.avg("revenue").alias("avg_order_value")
    )
    .orderBy("order_date", "store_id")
)

# ── write Gold by TABLE NAME ─────────────────────────
gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("adb_retail_dev.gold.sales_daily")

print(f"Gold rows: {gold.count()}")
display(gold)